In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [145]:
df = pd.read_csv("../data/raw/paper_leaks.csv")
df.head()

,incident_id,date,era,exam_name,conducting_body,body_type,area,leak_status,action_taken,note,arrests,convictions,aspirants_affected,linked_deaths,deaths_note,source_name,source_url,confidence
0,PL-0001,2004-04-11,UPA (2004-May2014),All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004,Central Board of Secondary Education (CBSE),Central,All India,Confirmed,Exam cancelled + Arrests-FIR + Probe (CBI/SIT),Physics and chemistry papers leaked in New Del...,2.0,NaN,NaN,NaN,NaN,The Tribune,https://www.tribuneindia.com/2004/20040412/mai...,High
1,PL-0002,2006-06-01,UPA (2004-May2014),Himachal Pradesh Combined Pre-Medical Test (HP...,Himachal Pradesh University,State,Himachal Pradesh,Confirmed,Retest + Arrests-FIR,HP-CPMT medical entrance paper leaked via a to...,NaN,NaN,NaN,NaN,NaN,The Tribune,https://www.tribuneindia.com/2006/20060831/him...,Medium
2,PL-0003,2008-01-01,UPA (2004-May2014),Vyapam Patwari Recruitment Examination 2008,Madhya Pradesh Professional Examination Board ...,State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Accused produced forged educational certificat...,NaN,10.0,NaN,NaN,NaN,The420.in,https://the420.in/vyapam-scam-cbi-court-patwar...,High
3,PL-0004,2009-07-08,UPA (2004-May2014),MP Pre-Medical Test (PMT) 2009,Vyapam (MPPEB),State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Paid solvers/impersonators sat the exam in pla...,NaN,NaN,114.0,NaN,NaN,The Free Press Journal,https://www.freepressjournal.in/education/indo...,High
4,PL-0005,2010-06-06,UPA (2004-May2014),RRB Mumbai 2010 recruitment exam (Assistant Lo...,"Railway Recruitment Board (RRB), Mumbai",Central,All India,Confirmed,Arrests-FIR + Probe (CBI/SIT),CBI registered a case on 15 June 2010 over the...,15.0,10.0,NaN,NaN,NaN,The Week,https://www.theweek.in/wire-updates/national/2...,High


In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   incident_id         110 non-null    object 
 1   date                110 non-null    object 
 2   era                 110 non-null    object 
 3   exam_name           110 non-null    object 
 4   conducting_body     110 non-null    object 
 5   body_type           110 non-null    object 
 6   area                110 non-null    object 
 7   leak_status         110 non-null    object 
 8   action_taken        110 non-null    object 
 9   note                110 non-null    object 
 10  arrests             63 non-null     float64
 11  convictions         10 non-null     float64
 12  aspirants_affected  44 non-null     float64
 13  linked_deaths       4 non-null      float64
 14  deaths_note         4 non-null      object 
 15  source_name         110 non-null    object 
 16  source_u

In [5]:
df.shape

(110, 18)

In [6]:
df.columns

Index(['incident_id', 'date', 'era', 'exam_name', 'conducting_body',
       'body_type', 'area', 'leak_status', 'action_taken', 'note', 'arrests',
       'convictions', 'aspirants_affected', 'linked_deaths', 'deaths_note',
       'source_name', 'source_url', 'confidence'],
      dtype='object')

In [ ]:
df.isnull().sum()

incident_id             0
date                    0
era                     0
exam_name               0
conducting_body         0
body_type               0
area                    0
leak_status             0
action_taken            0
note                    0
arrests                47
convictions           100
aspirants_affected     66
linked_deaths         106
deaths_note           106
source_name             0
source_url              0
confidence              0
dtype: int64

In [8]:

df["exam_category"] = ""

In [9]:
df[["exam_name", "exam_category"]].head(10)

,exam_name,exam_category
0,All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004,
1,Himachal Pradesh Combined Pre-Medical Test (HP...,
2,Vyapam Patwari Recruitment Examination 2008,
3,MP Pre-Medical Test (PMT) 2009,
4,RRB Mumbai 2010 recruitment exam (Assistant Lo...,
5,Vyapam Contract Teacher Eligibility Test (Samv...,
6,CBSE Class 10 & Class 12 Board Examinations 2011,
7,AIEEE (All India Engineering Entrance Examinat...,
8,Chhattisgarh Pre-Medical Test (CG-PMT) 2011,
9,Uttar Pradesh Teacher Eligibility Test (UPTET)...,


In [158]:
def get_exam_category(exam):
    exam = exam.lower()

    # Medical
    if any(x in exam for x in [
        "medical", "mbbs", "neet", "aipmt",
        "aiims", "nursing", "pharmacy",
        "cho", "pharmacist"
    ]):
        return "Medical"

    # Engineering
    elif any(x in exam for x in [
        "engineering", "aieee", "jee",
        "assistant engineer", "junior engineer",
        "ae", "je"
    ]):
        return "Engineering"

    # Banking
    elif any(x in exam for x in [
        "bank", "ibps", "sbi", "rbi",
        "probationary officer", "po"
    ]):
        return "Banking"

    # Civil Services
    elif any(x in exam for x in [
        "upsc", "civil service", "ias",
        "pcs", "uppcs", "mpsc",
        "gpsc", "bpsc", "ras"
    ]):
        return "Civil Services"

    # Police
    elif any(x in exam for x in [
        "police", "constable",
        "sub inspector", "si"
    ]):
        return "Police"

    # Teacher
    elif any(x in exam for x in [
        "teacher", "tet",
        "eligibility", "shikshak"
    ]):
        return "Teacher"

    # School Board
    elif any(x in exam for x in [
        "board", "class 10",
        "class 12", "intermediate"
    ]):
        return "School Board"

    # Railway
    elif any(x in exam for x in [
        "railway", "rrb"
    ]):
        return "Railway"

    # Defence
    elif any(x in exam for x in [
        "nda", "army",
        "navy", "air force",
        "defence"
    ]):
        return "Defence"

    # Forest
    elif any(x in exam for x in [
        "forest", "forest guard"
    ]):
        return "Forest"

    # Agriculture
    elif any(x in exam for x in [
        "agriculture", "agricultural"
    ]):
        return "Agriculture"

    # Judiciary
    elif any(x in exam for x in [
        "judicial", "judge"
    ]):
        return "Judiciary"

    # Higher Education
    elif any(x in exam for x in [
        "ugc", "net", "jrf"
    ]):
        return "Higher Education"

    # Food Inspector
    elif any(x in exam for x in [
        "food inspector"
    ]):
        return "Food Inspector"

    # Government Recruitment
    elif any(x in exam for x in [
        "recruitment", "clerk",
        "patwari", "lekhpal",
        "gram sachiv", "secretariat",
        "operator", "librarian",
        "graduate level", "cgl",
        "selection"
    ]):
        return "Government Recruitment"

    else:
        return "Other"

In [31]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   incident_id         110 non-null    object 
 1   date                110 non-null    object 
 2   era                 110 non-null    object 
 3   exam_name           110 non-null    object 
 4   conducting_body     110 non-null    object 
 5   body_type           110 non-null    object 
 6   area                110 non-null    object 
 7   leak_status         110 non-null    object 
 8   action_taken        110 non-null    object 
 9   note                110 non-null    object 
 10  arrests             63 non-null     float64
 11  convictions         10 non-null     float64
 12  aspirants_affected  44 non-null     float64
 13  linked_deaths       4 non-null      float64
 14  deaths_note         4 non-null      object 
 15  source_name         110 non-null    object 
 16  source_u

In [159]:
df["exam_category"] = df["exam_name"].apply(get_exam_category)

In [33]:
df[["exam_name", "exam_category"]].head(20)

,exam_name,exam_category
0,All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004,Medical
1,Himachal Pradesh Combined Pre-Medical Test (HP...,Medical
2,Vyapam Patwari Recruitment Examination 2008,Government Recruitment
3,MP Pre-Medical Test (PMT) 2009,Medical
4,RRB Mumbai 2010 recruitment exam (Assistant Lo...,Police
5,Vyapam Contract Teacher Eligibility Test (Samv...,Teacher
6,CBSE Class 10 & Class 12 Board Examinations 2011,School Board
7,AIEEE (All India Engineering Entrance Examinat...,Engineering
8,Chhattisgarh Pre-Medical Test (CG-PMT) 2011,Medical
9,Uttar Pradesh Teacher Eligibility Test (UPTET)...,Teacher


In [160]:
df["exam_category"].value_counts()

exam_category
Medical                   22
Government Recruitment    21
Banking                   13
Teacher                   11
Police                     8
Engineering                8
School Board               7
Civil Services             6
Forest                     5
Other                      2
Railway                    2
Food Inspector             1
Agriculture                1
Judiciary                  1
Defence                    1
Higher Education           1
Name: count, dtype: int64

In [19]:
df["exam_name"].sort_values().unique()

array(['AIEEE (All India Engineering Entrance Examination) 2011',
       'AIIMS NORCET-4 (Nursing Officer Recruitment)',
       'AIIMS Post-Graduate (MD/MS) Entrance Examination 2012',
       'AIPMT 2015 (All India Pre-Medical Test)',
       'Agricultural Development Officer & combined competitive recruitment (2013–2014)',
       'All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004',
       'Army Soldier General Duty Common Entrance Exam 2021',
       'Assistant Engineer (Civil) exam',
       'Assistant Engineer (Civil) recruitment exam (also voided Group-I Prelims, AEE, DAO)',
       'Assistant Engineer / Junior Engineer (AE/JE) recruitment exam',
       'BPSC 67th Combined Competitive Exam (Prelims)',
       'BPSC 70th Combined Competitive Exam (Prelims)',
       'BPSC TRE 3.0 School Teacher Recruitment Exam',
       'BSSC 1st Inter-Level Combined (Clerical-grade) Competitive Exam',
       'BSSC 3rd Graduate Level (CGL) Combined Competitive Prelims 2022',
       'Bihar Police Constable 

In [161]:
df[df["exam_category"] == "Other"][["exam_name"]]

,exam_name
31,BSSC 1st Inter-Level Combined (Clerical-grade)...
44,Sub-Inspector (unarmed branch) written exam


In [162]:
df[df["exam_category"] == "Government Recruitment"][["exam_name"]]

,exam_name
2,Vyapam Patwari Recruitment Examination 2008
20,SSC Combined Graduate Level (CGL) Examination ...
29,Village Panchayat Development Officer (VPDO/VD...
36,SSC CGL 2017 Tier-II
40,UPSSSC Tubewell Operator (Nalkoop Chalak) Recr...
41,Non-Secretariat Clerk recruitment exam
42,Librarian Grade III recruitment exam
48,HSSC Gram Sachiv (Panchayat Secretary) exam
56,Patwari (village revenue officer) recruitment ...
58,Graduate-level (Snatak) recruitment exam


In [163]:
df[df["exam_category"]=="Government Recruitment"][["exam_name"]]

,exam_name
2,Vyapam Patwari Recruitment Examination 2008
20,SSC Combined Graduate Level (CGL) Examination ...
29,Village Panchayat Development Officer (VPDO/VD...
36,SSC CGL 2017 Tier-II
40,UPSSSC Tubewell Operator (Nalkoop Chalak) Recr...
41,Non-Secretariat Clerk recruitment exam
42,Librarian Grade III recruitment exam
48,HSSC Gram Sachiv (Panchayat Secretary) exam
56,Patwari (village revenue officer) recruitment ...
58,Graduate-level (Snatak) recruitment exam


In [164]:
df[df["exam_category"]=="Other"][["exam_name"]]

,exam_name
31,BSSC 1st Inter-Level Combined (Clerical-grade)...
44,Sub-Inspector (unarmed branch) written exam


In [165]:
df.loc[
    df["exam_name"].str.contains("BSSC 1st Inter-Level", case=False, na=False),
    "exam_category"
] = "Government Recruitment"

In [166]:
df.loc[
    df["exam_name"].str.contains("Sub-Inspector", case=False, na=False),
    "exam_category"
] = "Police"

In [168]:
df["exam_category"].value_counts()

exam_category
Medical                   22
Government Recruitment    22
Police                    11
Teacher                   11
Banking                   11
Engineering                8
School Board               7
Civil Services             6
Forest                     5
Railway                    2
Food Inspector             1
Agriculture                1
Judiciary                  1
Defence                    1
Higher Education           1
Name: count, dtype: int64

In [ ]:
df[df["exam_category"] == "Forest"][["exam_name"]]

,exam_name
0,All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004
1,Himachal Pradesh Combined Pre-Medical Test (HP...
3,MP Pre-Medical Test (PMT) 2009
8,Chhattisgarh Pre-Medical Test (CG-PMT) 2011
11,MP Pre-PG (Post-Graduate Medical) Test 2012
13,AIIMS Post-Graduate (MD/MS) Entrance Examinati...
15,Jammu & Kashmir Common Entrance Test (JKCET) 2...
16,MP Pre-Medical Test (PMT) 2012
21,MP Pre-Medical Test (PMT) 2013
26,AIPMT 2015 (All India Pre-Medical Test)


In [41]:
df[df["exam_category"] == "Engineering"][["exam_name"]]

,exam_name
7,AIEEE (All India Engineering Entrance Examinat...
39,UPPCL Junior Engineer / Technician online recr...
46,Junior Engineer (JEN) recruitment exam
69,Assistant Engineer / Junior Engineer (AE/JE) r...
71,Assistant Engineer (Civil) exam
79,Assistant Engineer (Civil) recruitment exam (a...
84,Junior Engineer (Civil) main written exam (Com...
106,Jharkhand Excise Constable Competitive Exam (J...


In [42]:
df[df["exam_category"] == "Forest"][["exam_name"]]

,exam_name
50,"Forest Guard, Secretariat Security Guard and a..."
63,Forest Guard recruitment exam
72,Forest Guard recruitment exam
82,MP Forest Guard (Vanrakshak) Recruitment Exam ...
94,MPPSC State Service & State Forest Service Pre...


In [169]:
df["medical_type"] = "Not Medical"

In [170]:
df.loc[
    df["exam_name"].str.contains("NEET-UG", case=False, na=False),
    "medical_type"
] = "NEET UG"

In [171]:
df.loc[
    df["exam_name"].str.contains("NEET-PG", case=False, na=False),
    "medical_type"
] = "NEET PG"

In [172]:
df.loc[
    df["exam_name"].str.contains("AIPMT", case=False, na=False),
    "medical_type"
] = "AIPMT"

In [173]:
df.loc[
    df["exam_name"].str.contains("AIIMS", case=False, na=False),
    "medical_type"
] = "AIIMS"

In [174]:
df.loc[
    df["exam_name"].str.contains("PMT", case=False, na=False),
    "medical_type"
] = "State PMT"

In [175]:
df.loc[
    df["exam_name"].str.contains("Medical Officer", case=False, na=False),
    "medical_type"
] = "Medical Officer"

In [176]:
df.loc[
    df["exam_name"].str.contains("Nursing", case=False, na=False),
    "medical_type"
] = "Nursing"

In [177]:
df.loc[
    df["exam_name"].str.contains("Pharmacist", case=False, na=False),
    "medical_type"
] = "Pharmacist"

In [178]:
df.loc[
    df["exam_name"].str.contains("Community Health Officer|CHO", case=False, na=False),
    "medical_type"
] = "Community Health Officer"

In [179]:
df.loc[
    df["exam_name"].str.contains("JKCET", case=False, na=False),
    "medical_type"
] = "State Medical Entrance"

In [180]:
df.loc[
    df["exam_name"].str.contains("HP", case=False, na=False),
    "medical_type"
] = "State Medical Entrance"

In [181]:
df["engineering_type"] = "Not Engineering"

In [182]:
df.loc[
    df["exam_name"].str.contains("AIEEE", case=False, na=False),
    "engineering_type"
] = "Engineering Entrance"

In [57]:
df.loc[
    df["exam_name"].str.contains("Junior Engineer|JEN|JE", case=False, na=False),
    "engineering_type"
] = "Junior Engineer Recruitment"

In [183]:
df.loc[
    df["exam_name"].str.contains("Assistant Engineer|AE", case=False, na=False),
    "engineering_type"
] = "Assistant Engineer Recruitment"

In [184]:
df["forest_type"] = "Not Forest"

In [185]:
df.loc[
    df["exam_name"].str.contains("Forest Guard", case=False, na=False),
    "forest_type"
] = "Forest Guard"

In [186]:
df.loc[
    df["exam_name"].str.contains("Forest Service", case=False, na=False),
    "forest_type"
] = "State Forest Service"

In [187]:
df.duplicated().sum()

0

In [188]:
df.isnull().sum()

incident_id             0
date                    0
era                     0
exam_name               0
conducting_body         0
body_type               0
area                    0
leak_status             0
action_taken            0
note                    0
arrests                47
convictions           100
aspirants_affected     66
linked_deaths         106
deaths_note           106
source_name             0
source_url              0
confidence              0
arrests_status          0
conviction_status       0
deaths_status           0
exam_category           0
medical_type            0
engineering_type        0
forest_type             0
dtype: int64

In [189]:
df["year"] = pd.to_datetime(df["date"]).dt.year

In [190]:
df["quarter"] = "Q" + pd.to_datetime(df["date"]).dt.quarter.astype(str)

In [191]:
df["state"] = df["area"].str.strip()

In [192]:
region_map = {
    "Andhra Pradesh": "South",
    "Arunachal Pradesh": "North-East",
    "Assam": "North-East",
    "Bihar": "East",
    "Chhattisgarh": "Central",
    "Delhi": "North",
    "Goa": "West",
    "Gujarat": "West",
    "Haryana": "North",
    "Himachal Pradesh": "North",
    "Jharkhand": "East",
    "Karnataka": "South",
    "Kerala": "South",
    "Madhya Pradesh": "Central",
    "Maharashtra": "West",
    "Manipur": "North-East",
    "Meghalaya": "North-East",
    "Mizoram": "North-East",
    "Nagaland": "North-East",
    "Odisha": "East",
    "Punjab": "North",
    "Rajasthan": "North",
    "Sikkim": "North-East",
    "Tamil Nadu": "South",
    "Telangana": "South",
    "Tripura": "North-East",
    "Uttar Pradesh": "North",
    "Uttarakhand": "North",
    "West Bengal": "East",
    "Jammu & Kashmir": "North",
    "Ladakh": "North",
    "All India": "National"
}

In [193]:
df["region"] = df["state"].map(region_map)

In [195]:
df["region"] = df["region"].fillna("Unknown")

In [196]:
national_keywords = [
    "National",
    "All India",
    "NEET",
    "AIPMT",
    "AIIMS",
    "JEE",
    "AIEEE",
    "GATE",
    "UPSC",
    "SSC",
    "CBSE",
    "NTA"
]
df["organization_level"] = "State"
df.loc[
    df["exam_name"].str.contains("|".join(national_keywords), case=False, na=False),
    "organization_level"
] = "National"

In [197]:
df["severity"] = pd.cut(
    df["aspirants_affected"],
    bins=[0,10000,50000,float("inf")],
    labels=["Low","Medium","High"]
)
df["severity"] = df["severity"].cat.add_categories("Unknown")
df["severity"] = df["severity"].fillna("Unknown")

In [198]:
df["affected_group"] = pd.cut(
    df["aspirants_affected"],
    bins=[0,10000,50000,float("inf")],
    labels=["<10K","10K-50K",">50K"]
)
df["affected_group"] = df["affected_group"].cat.add_categories("Unknown")
df["affected_group"] = df["affected_group"].fillna("Unknown")

In [199]:
df["exam_level"] = "Other"

df.loc[df["exam_name"].str.contains("UG|Undergraduate|AIEEE|JEE|NEET-UG|AIPMT|PMT", case=False, na=False), "exam_level"] = "UG"

df.loc[df["exam_name"].str.contains("PG|Post-Graduate|NEET-PG|MD|MS", case=False, na=False), "exam_level"] = "PG"

df.loc[df["exam_name"].str.contains("Recruitment|Officer|Constable|Inspector|Engineer|Teacher|Clerk|Assistant|Guard|Police|SSC|PSC|CHO|NORCET", case=False, na=False), "exam_level"] = "Recruitment"

df.loc[df["exam_name"].str.contains("Class 10|Class 12|Board|Intermediate|School", case=False, na=False), "exam_level"] = "School"

df.loc[df["exam_name"].str.contains("Civil Services|Judiciary|Bar Council|Professional", case=False, na=False), "exam_level"] = "Professional"

In [200]:
# conducting_body_category
df["conducting_body_category"] = "Other"

df.loc[df["conducting_body"].str.contains("UPSC", case=False, na=False), "conducting_body_category"] = "UPSC"

df.loc[df["conducting_body"].str.contains("SSC", case=False, na=False), "conducting_body_category"] = "SSC"

df.loc[df["conducting_body"].str.contains("PSC", case=False, na=False), "conducting_body_category"] = "State PSC"

df.loc[df["conducting_body"].str.contains("University|AIIMS", case=False, na=False), "conducting_body_category"] = "University"

df.loc[df["conducting_body"].str.contains("Board|CBSE", case=False, na=False), "conducting_body_category"] = "Board"

df.loc[df["conducting_body"].str.contains("NTA", case=False, na=False), "conducting_body_category"] = "NTA"

In [201]:
# exam_mode column
df["exam_mode"] = "Unknown"

df.loc[df["exam_name"].str.contains("online|computer", case=False, na=False), "exam_mode"] = "Online"

df.loc[df["exam_name"].str.contains("written|offline|board", case=False, na=False), "exam_mode"] = "Offline"

In [202]:
# decade column
df["decade"] = (df["year"] // 10) * 10
df["decade"] = df["decade"].astype(str) + "s"

In [203]:
# state_code column
state_code = {
    "Andhra Pradesh":"AP",
    "Arunachal Pradesh":"AR",
    "Assam":"AS",
    "Bihar":"BR",
    "Chhattisgarh":"CG",
    "Delhi":"DL",
    "Goa":"GA",
    "Gujarat":"GJ",
    "Haryana":"HR",
    "Himachal Pradesh":"HP",
    "Jharkhand":"JH",
    "Jammu & Kashmir":"JK",
    "Karnataka":"KA",
    "Kerala":"KL",
    "Madhya Pradesh":"MP",
    "Maharashtra":"MH",
    "Manipur":"MN",
    "Meghalaya":"ML",
    "Mizoram":"MZ",
    "Nagaland":"NL",
    "Odisha":"OD",
    "Punjab":"PB",
    "Rajasthan":"RJ",
    "Sikkim":"SK",
    "Tamil Nadu":"TN",
    "Telangana":"TS",
    "Tripura":"TR",
    "Uttar Pradesh":"UP",
    "Uttarakhand":"UK",
    "West Bengal":"WB",
    "All India":"IND"
}

df["state_code"] = df["state"].map(state_code)

In [205]:
# region_code column
region_code = {
    "North":"N",
    "South":"S",
    "East":"E",
    "West":"W",
    "Central":"C",
    "North-East":"NE",
    "National":"IND",
    "Unknown":"UNK"
}

df["region_code"] = df["region"].map(region_code)

In [206]:
# affected_category column
df["affected_category"] = pd.cut(
    df["aspirants_affected"],
    bins=[0,5000,25000,100000,float("inf")],
    labels=["Low","Moderate","High","Very High"]
)

df["affected_category"] = (
    df["affected_category"]
    .cat.add_categories("Unknown")
    .fillna("Unknown")
)

In [81]:
# arrests_status column
df["arrests_status"] = df["arrests"].apply(
    lambda x: "Arrest Made" if pd.notna(x) and x > 0 else "No Arrest"
)

In [83]:
# conviction_status column
df["conviction_status"] = df["convictions"].apply(
    lambda x: "Convicted" if pd.notna(x) and x > 0 else "Not Convicted"
)

In [ ]:
# deaths_status column
df["deaths_status"] = df["linked_deaths"].apply(
    lambda x: "Yes" if pd.notna(x) and x > 0 else "No"
)

In [207]:
# leak_confirmed column
df["leak_confirmed"] = df["confidence"].replace({
    "High": "Confirmed",
    "Medium": "Suspected",
    "Low": "Suspected"
})

In [208]:
# repeat_exam column
exam_counts = df["exam_name"].value_counts()

df["repeat_exam"] = df["exam_name"].map(exam_counts)

df["repeat_exam"] = df["repeat_exam"].apply(
    lambda x: "Yes" if x > 1 else "No"
)

In [209]:
new_columns = [
    "exam_level",
    "conducting_body_category",
    "exam_mode",
    "decade",
    "state_code",
    "region_code",
    "affected_category",
    "arrests_status",
    "conviction_status",
    "deaths_status",
    "leak_confirmed",
    "repeat_exam"
]

df[new_columns].head()

,exam_level,conducting_body_category,exam_mode,decade,state_code,region_code,affected_category,arrests_status,conviction_status,deaths_status,leak_confirmed,repeat_exam
0,UG,Board,Unknown,2000s,IND,IND,Unknown,Arrests Made,Not Reported,Not Reported,Confirmed,No
1,UG,University,Unknown,2000s,HP,N,Unknown,Not Reported,Not Reported,Not Reported,Suspected,No
2,Recruitment,Board,Unknown,2000s,MP,C,Unknown,Not Reported,Convictions Reported,Not Reported,Confirmed,No
3,UG,Other,Unknown,2000s,MP,C,Low,Not Reported,Not Reported,Not Reported,Confirmed,No
4,Recruitment,Board,Unknown,2010s,IND,IND,Unknown,Arrests Made,Convictions Reported,Not Reported,Confirmed,No


In [90]:
df[df["conducting_body_category"] == "Other"][["conducting_body"]].drop_duplicates()

,conducting_body
3,Vyapam (MPPEB)
33,Punjab & Haryana High Court
39,Uttar Pradesh Power Corporation Ltd (UPPCL)
43,Maharashtra State Council of Examination (MSCE)
47,Railway Recruitment Cell (Western Railway)
49,Indian Army
55,Karnataka State Police / Recruitment Wing
60,Maharashtra Housing and Area Development Autho...
61,Tamil Nadu School Education Department
62,Rajasthan High Court


In [91]:
df[df["exam_mode"] == "Unknown"][["exam_name"]]

,exam_name
0,All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004
1,Himachal Pradesh Combined Pre-Medical Test (HP...
2,Vyapam Patwari Recruitment Examination 2008
3,MP Pre-Medical Test (PMT) 2009
4,RRB Mumbai 2010 recruitment exam (Assistant Lo...
...,...
103,Group-B recruitment exam (408 posts incl Naib ...
106,Jharkhand Excise Constable Competitive Exam (J...
107,NEET-UG 2026
108,HTET Level-3 / PGT (Haryana Teacher Eligibilit...


In [210]:
df["exam_mode"] = "Offline"

df.loc[
    df["exam_name"].str.contains(
        "online|computer|CBT|computer based|computer-based",
        case=False,
        na=False
    ),
    "exam_mode"
] = "Online"

In [211]:
df.loc[
    df["conducting_body"].str.contains("Vyapam|MPPEB", case=False, na=False),
    "conducting_body_category"
] = "State Recruitment Board"

In [213]:
df.loc[
    df["conducting_body"].str.contains("Vyapam|MPPEB", case=False, na=False),
    "conducting_body_category"
] = "State Recruitment Board"
df.loc[
    df["conducting_body"].str.contains("High Court", case=False, na=False),
    "conducting_body_category"
] = "Judiciary"
df.loc[
    df["conducting_body"].str.contains("UPPCL|MHADA|Revenue", case=False, na=False),
    "conducting_body_category"
] = "Government Department"
df.loc[
    df["conducting_body"].str.contains("MSCE", case=False, na=False),
    "conducting_body_category"
] = "Board"
df.loc[
    df["conducting_body"].str.contains("Railway Recruitment", case=False, na=False),
    "conducting_body_category"
] = "Railway"
df.loc[
    df["conducting_body"].str.contains("Indian Army", case=False, na=False),
    "conducting_body_category"
] = "Defence"
df.loc[
    df["conducting_body"].str.contains("Police", case=False, na=False),
    "conducting_body_category"
] = "Police Recruitment"
df.loc[
    df["conducting_body"].str.contains("School Education", case=False, na=False),
    "conducting_body_category"
] = "Education Department"
df.loc[
    df["conducting_body"].str.contains("NBEMS", case=False, na=False),
    "conducting_body_category"
] = "Medical Board"
df.loc[
    df["conducting_body"].str.contains("State Health Society|NHM", case=False, na=False),
    "conducting_body_category"
] = "Health Department"

In [216]:
df["conducting_body_category"].value_counts()

conducting_body_category
Board                      32
SSC                        19
State PSC                  17
State Recruitment Board    11
University                  7
Police Recruitment          6
Education Department        4
NTA                         4
Government Department       3
Railway                     2
Judiciary                   2
Defence                     1
Medical Board               1
Health Department           1
Name: count, dtype: int64

In [214]:
df["exam_mode"] = "Unknown"

In [217]:
df.loc[
    df["exam_name"].str.contains(
        "online|CBT|computer|computer based|computer-based",
        case=False,
        na=False
    ),
    "exam_mode"
] = "Online"

In [218]:
df["exam_mode"].value_counts()

exam_mode
Unknown    108
Online       2
Name: count, dtype: int64

In [220]:
df.head()

,incident_id,date,era,exam_name,conducting_body,body_type,area,leak_status,action_taken,note,...,affected_group,exam_level,conducting_body_category,exam_mode,decade,state_code,region_code,affected_category,leak_confirmed,repeat_exam
0,PL-0001,2004-04-11,UPA (2004-May2014),All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004,Central Board of Secondary Education (CBSE),Central,All India,Confirmed,Exam cancelled + Arrests-FIR + Probe (CBI/SIT),Physics and chemistry papers leaked in New Del...,...,Unknown,UG,Board,Unknown,2000s,IND,IND,Unknown,Confirmed,No
1,PL-0002,2006-06-01,UPA (2004-May2014),Himachal Pradesh Combined Pre-Medical Test (HP...,Himachal Pradesh University,State,Himachal Pradesh,Confirmed,Retest + Arrests-FIR,HP-CPMT medical entrance paper leaked via a to...,...,Unknown,UG,University,Unknown,2000s,HP,N,Unknown,Suspected,No
2,PL-0003,2008-01-01,UPA (2004-May2014),Vyapam Patwari Recruitment Examination 2008,Madhya Pradesh Professional Examination Board ...,State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Accused produced forged educational certificat...,...,Unknown,Recruitment,State Recruitment Board,Unknown,2000s,MP,C,Unknown,Confirmed,No
3,PL-0004,2009-07-08,UPA (2004-May2014),MP Pre-Medical Test (PMT) 2009,Vyapam (MPPEB),State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Paid solvers/impersonators sat the exam in pla...,...,<10K,UG,State Recruitment Board,Unknown,2000s,MP,C,Low,Confirmed,No
4,PL-0005,2010-06-06,UPA (2004-May2014),RRB Mumbai 2010 recruitment exam (Assistant Lo...,"Railway Recruitment Board (RRB), Mumbai",Central,All India,Confirmed,Arrests-FIR + Probe (CBI/SIT),CBI registered a case on 15 June 2010 over the...,...,Unknown,Recruitment,Railway,Unknown,2010s,IND,IND,Unknown,Confirmed,No


In [221]:
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

1. incident_id
2. date
3. era
4. exam_name
5. conducting_body
6. body_type
7. area
8. leak_status
9. action_taken
10. note
11. arrests
12. convictions
13. aspirants_affected
14. linked_deaths
15. deaths_note
16. source_name
17. source_url
18. confidence
19. arrests_status
20. conviction_status
21. deaths_status
22. exam_category
23. medical_type
24. engineering_type
25. forest_type
26. year
27. quarter
28. state
29. region
30. organization_level
31. severity
32. affected_group
33. exam_level
34. conducting_body_category
35. exam_mode
36. decade
37. state_code
38. region_code
39. affected_category
40. leak_confirmed
41. repeat_exam


In [103]:
df["incident_id"].duplicated().sum()

0

In [104]:
df.shape

(110, 41)

In [105]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   incident_id               110 non-null    object  
 1   date                      110 non-null    object  
 2   era                       110 non-null    object  
 3   exam_name                 110 non-null    object  
 4   conducting_body           110 non-null    object  
 5   body_type                 110 non-null    object  
 6   area                      110 non-null    object  
 7   leak_status               110 non-null    object  
 8   action_taken              110 non-null    object  
 9   note                      110 non-null    object  
 10  arrests                   63 non-null     float64 
 11  convictions               10 non-null     float64 
 12  aspirants_affected        44 non-null     float64 
 13  linked_deaths             4 non-null      float64 

In [106]:
df["date"] = pd.to_datetime(df["date"])

In [107]:
df["date"].dtype

dtype('<M8[ns]')

In [108]:
df.isnull().sum().sort_values(ascending=False)

linked_deaths               106
deaths_note                 106
convictions                 100
aspirants_affected           66
arrests                      47
state_code                   43
region                        0
organization_level            0
severity                      0
affected_group                0
exam_level                    0
conducting_body_category      0
exam_mode                     0
incident_id                   0
quarter                       0
decade                        0
region_code                   0
affected_category             0
arrests_status                0
conviction_status             0
deaths_status                 0
leak_confirmed                0
state                         0
engineering_type              0
year                          0
action_taken                  0
era                           0
exam_name                     0
conducting_body               0
body_type                     0
area                          0
leak_sta

In [109]:
df[df["state_code"].isna()]["state"].value_counts()

state
Uttar Pradesh (statewide)                             3
Uttarakhand (Haridwar)                                3
Rajasthan (Jodhpur)                                   2
Jharkhand (Ranchi)                                    2
Delhi (NCT of Delhi)                                  2
Punjab (Bathinda highlighted)                         1
Bihar (Bapu Exam Centre, Patna)                       1
Bihar (Patna and other centres)                       1
Jharkhand (Jamtara/Chatra/Dhanbad)                    1
Bihar (statewide)                                     1
Bihar (Ara, Bhojpur)                                  1
Bihar (37 districts)                                  1
Himachal Pradesh (Hamirpur)                           1
Rajasthan (Udaipur)                                   1
Bihar (528 centres, 38 districts)                     1
Rajasthan (Rajsamand)                                 1
Maharashtra (Nagpur)                                  1
Punjab (Patiala; centres statewide)       

In [110]:
df["state"] = df["state"].str.replace(r"\s*\(.*?\)", "", regex=True)

In [111]:
df["state"] = df["state"].replace({
    "Andaman & Nicobar Islands": "Andaman and Nicobar Islands"
})

In [112]:
df["state_code"] = df["state"].map(state_code)

In [113]:
df[df["state_code"].isna()][["state"]].drop_duplicates()

,state
6,Andaman and Nicobar Islands
31,Bihar / Jharkhand


In [115]:
state_code["Andaman and Nicobar Islands"] = "AN"
state_code["Bihar / Jharkhand"] = "BR/JH"
df["state_code"] = df["state"].map(state_code)

In [116]:
df[df["state_code"].isna()]

,incident_id,date,era,exam_name,conducting_body,body_type,area,leak_status,action_taken,note,...,exam_mode,decade,state_code,region_code,affected_category,arrests_status,conviction_status,deaths_status,leak_confirmed,repeat_exam


In [117]:
df.isnull().sum()

incident_id                   0
date                          0
era                           0
exam_name                     0
conducting_body               0
body_type                     0
area                          0
leak_status                   0
action_taken                  0
note                          0
arrests                      47
convictions                 100
aspirants_affected           66
linked_deaths               106
deaths_note                 106
source_name                   0
source_url                    0
confidence                    0
exam_category                 0
medical_type                  0
engineering_type              0
forest_type                   0
year                          0
quarter                       0
state                         0
region                        0
organization_level            0
severity                      0
affected_group                0
exam_level                    0
conducting_body_category      0
exam_mod

In [128]:
# 1. Arrests
df["arrests"] = df["arrests"].fillna("Not Reported")

# 2. Convictions
df["convictions"] = df["convictions"].fillna("Not Reported")

# 3. Linked Deaths
df["linked_deaths"] = df["linked_deaths"].fillna("Not Reported")

# 4. Deaths Note
df["deaths_note"] = df["deaths_note"].fillna("Not Reported")

In [140]:
df.isnull().sum()

incident_id                  0
date                         0
era                          0
exam_name                    0
conducting_body              0
body_type                    0
area                         0
leak_status                  0
action_taken                 0
note                         0
arrests                      0
convictions                  0
aspirants_affected          66
linked_deaths                0
deaths_note                  0
source_name                  0
source_url                   0
confidence                   0
exam_category                0
medical_type                 0
engineering_type             0
forest_type                  0
year                         0
quarter                      0
state                        0
region                       0
organization_level           0
severity                     0
affected_group               0
exam_level                   0
conducting_body_category     0
exam_mode                    0
decade  

In [142]:
df.shape




(110, 41)

In [143]:
df["arrests"].value_counts(dropna=False)

arrests
0      48
3       9
12      4
5       4
2       3
10      3
17      2
37      2
54      2
6       2
30      2
18      2
20      2
9       2
7       2
1       2
4       1
26      1
266     1
148     1
15      1
19      1
44      1
21      1
8       1
88      1
70      1
11      1
14      1
23      1
38      1
167     1
24      1
13      1
164     1
Name: count, dtype: int64

In [146]:
df.isnull().sum()

incident_id             0
date                    0
era                     0
exam_name               0
conducting_body         0
body_type               0
area                    0
leak_status             0
action_taken            0
note                    0
arrests                47
convictions           100
aspirants_affected     66
linked_deaths         106
deaths_note           106
source_name             0
source_url              0
confidence              0
dtype: int64

In [149]:
df["convictions"].head(10)
# 3. Check linked deaths


0     NaN
1     NaN
2    10.0
3     NaN
4    10.0
5     NaN
6     4.0
7     NaN
8     NaN
9     NaN
Name: convictions, dtype: float64

In [150]:
df["linked_deaths"].head(10)

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: linked_deaths, dtype: float64

In [152]:
[col for col in df.columns if "status" in col]

['leak_status']

In [153]:
df["arrests_status"] = df["arrests"].apply(
    lambda x: "Not Reported" if pd.isna(x)
    else "Arrests Made" if x > 0
    else "No Arrests"
)
df["conviction_status"] = df["convictions"].apply(
    lambda x: "Not Reported" if pd.isna(x)
    else "Convictions Reported" if x > 0
    else "No Convictions"
)
df["deaths_status"] = df["linked_deaths"].apply(
    lambda x: "Not Reported" if pd.isna(x)
    else "Deaths Reported" if x > 0
    else "No Deaths"
)

In [154]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   incident_id         110 non-null    object 
 1   date                110 non-null    object 
 2   era                 110 non-null    object 
 3   exam_name           110 non-null    object 
 4   conducting_body     110 non-null    object 
 5   body_type           110 non-null    object 
 6   area                110 non-null    object 
 7   leak_status         110 non-null    object 
 8   action_taken        110 non-null    object 
 9   note                110 non-null    object 
 10  arrests             63 non-null     float64
 11  convictions         10 non-null     float64
 12  aspirants_affected  44 non-null     float64
 13  linked_deaths       4 non-null      float64
 14  deaths_note         4 non-null      object 
 15  source_name         110 non-null    object 
 16  source_u

In [222]:
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

1. incident_id
2. date
3. era
4. exam_name
5. conducting_body
6. body_type
7. area
8. leak_status
9. action_taken
10. note
11. arrests
12. convictions
13. aspirants_affected
14. linked_deaths
15. deaths_note
16. source_name
17. source_url
18. confidence
19. arrests_status
20. conviction_status
21. deaths_status
22. exam_category
23. medical_type
24. engineering_type
25. forest_type
26. year
27. quarter
28. state
29. region
30. organization_level
31. severity
32. affected_group
33. exam_level
34. conducting_body_category
35. exam_mode
36. decade
37. state_code
38. region_code
39. affected_category
40. leak_confirmed
41. repeat_exam


In [223]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   incident_id               110 non-null    object  
 1   date                      110 non-null    object  
 2   era                       110 non-null    object  
 3   exam_name                 110 non-null    object  
 4   conducting_body           110 non-null    object  
 5   body_type                 110 non-null    object  
 6   area                      110 non-null    object  
 7   leak_status               110 non-null    object  
 8   action_taken              110 non-null    object  
 9   note                      110 non-null    object  
 10  arrests                   63 non-null     float64 
 11  convictions               10 non-null     float64 
 12  aspirants_affected        44 non-null     float64 
 13  linked_deaths             4 non-null      float64 

In [224]:
df.isnull().sum()

incident_id                   0
date                          0
era                           0
exam_name                     0
conducting_body               0
body_type                     0
area                          0
leak_status                   0
action_taken                  0
note                          0
arrests                      47
convictions                 100
aspirants_affected           66
linked_deaths               106
deaths_note                 106
source_name                   0
source_url                    0
confidence                    0
arrests_status                0
conviction_status             0
deaths_status                 0
exam_category                 0
medical_type                  0
engineering_type              0
forest_type                   0
year                          0
quarter                       0
state                         0
region                        0
organization_level            0
severity                      0
affected

In [226]:
df["date"] = pd.to_datetime(df["date"])

In [228]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   incident_id               110 non-null    object        
 1   date                      110 non-null    datetime64[ns]
 2   era                       110 non-null    object        
 3   exam_name                 110 non-null    object        
 4   conducting_body           110 non-null    object        
 5   body_type                 110 non-null    object        
 6   area                      110 non-null    object        
 7   leak_status               110 non-null    object        
 8   action_taken              110 non-null    object        
 9   note                      110 non-null    object        
 10  arrests                   63 non-null     float64       
 11  convictions               10 non-null     float64       
 12  aspirants_affected    

In [257]:
df.to_csv("../data/cleaned/paper_leaks_cleaned.csv", index=False)

In [258]:
df[df["exam_category"]=="Other"][["exam_name"]]

,exam_name


In [ ]:
df[df["exam_category"]=="Other"][["exam_name"]]

,exam_name
31,BSSC 1st Inter-Level Combined (Clerical-grade)...
44,Sub-Inspector (unarmed branch) written exam


In [232]:
df[df["exam_category"]=="Other"][["exam_name"]]

,exam_name


In [259]:
import os
os.path.exists("../data/cleaned/paper_leaks_cleaned.csv")

True

In [235]:
df[df["state"] == "India"][["exam_name", "state", "state_code"]]

,exam_name,state,state_code


In [237]:
df["state"].value_counts()

state
All India                                             16
Madhya Pradesh                                        13
Uttar Pradesh                                          6
Gujarat                                                5
Maharashtra                                            4
Uttarakhand (Haridwar)                                 3
Chhattisgarh                                           3
Uttar Pradesh (statewide)                              3
Delhi (NCT of Delhi)                                   2
Rajasthan (Jodhpur)                                    2
Himachal Pradesh                                       2
Haryana                                                2
Jharkhand (Ranchi)                                     2
Uttarakhand                                            2
Rajasthan                                              2
Assam                                                  2
Karnataka                                              2
Jharkhand (Jamtara/Chatra

In [239]:
df.loc[df["state"] == "All India", "state_code"] = "IN"

df.loc[df["state"] == "All India", "region_code"] = "National"

In [241]:
df[df["state"] == "All India"][["state", "state_code", "region_code"]].head()

,state,state_code,region_code
0,All India,IN,National
4,All India,IN,National
20,All India,IN,National
26,All India,IN,National
30,All India,IN,National


In [242]:
df["state"] = df["state"].str.replace(r"\s*\(.*?\)", "", regex=True)

In [243]:
df["state"] = df["state"].replace({
    "Uttar Pradesh statewide": "Uttar Pradesh",
    "Uttar Pradesh ": "Uttar Pradesh",
    "Andaman & Nicobar Islands": "Andaman and Nicobar Islands",
    "Bihar / Jharkhand": "Bihar/Jharkhand"
})

In [244]:
df.loc[df["state"] == "All India", "state_code"] = "IN"
df.loc[df["state"] == "All India", "region_code"] = "National"

In [245]:
df["state"].value_counts()

state
All India                      16
Madhya Pradesh                 13
Uttar Pradesh                  12
Rajasthan                      11
Bihar                           6
Uttarakhand                     6
Maharashtra                     5
Gujarat                         5
Haryana                         5
Punjab                          4
Jharkhand                       4
Himachal Pradesh                3
Delhi                           3
Chhattisgarh                    3
Assam                           2
Karnataka                       2
Manipur                         1
Jammu & Kashmir                 1
Bihar/Jharkhand                 1
Tamil Nadu                      1
Andhra Pradesh                  1
Andaman and Nicobar Islands     1
Arunachal Pradesh               1
Telangana                       1
Odisha                          1
West Bengal                     1
Name: count, dtype: int64

In [249]:
df.loc[df["state"] == "All India", "state_code"] = "IN"
df.loc[df["state"] == "All India", "region_code"] = "NAT"

In [251]:
df[df["state"] == "All India"][["state", "region", "state_code", "region_code"]].head()

,state,region,state_code,region_code
0,All India,National,IN,NAT
4,All India,National,IN,NAT
20,All India,National,IN,NAT
26,All India,National,IN,NAT
30,All India,National,IN,NAT


In [253]:
df[df["organization_level"] == "State"][
    ["exam_name", "conducting_body", "organization_level"]
]

,exam_name,conducting_body,organization_level
1,Himachal Pradesh Combined Pre-Medical Test (HP...,Himachal Pradesh University,State
2,Vyapam Patwari Recruitment Examination 2008,Madhya Pradesh Professional Examination Board ...,State
3,MP Pre-Medical Test (PMT) 2009,Vyapam (MPPEB),State
4,RRB Mumbai 2010 recruitment exam (Assistant Lo...,"Railway Recruitment Board (RRB), Mumbai",State
5,Vyapam Contract Teacher Eligibility Test (Samv...,Madhya Pradesh Professional Examination Board ...,State
...,...,...,...
104,Class 12 Chemistry board examination,Maharashtra State Board of Secondary & Higher ...,State
105,CGBSE Class 12 Hindi Board Examination 2026,Chhattisgarh Board of Secondary Education (CGBSE),State
106,Jharkhand Excise Constable Competitive Exam (J...,Jharkhand Staff Selection Commission (JSSC),State
108,HTET Level-3 / PGT (Haryana Teacher Eligibilit...,Haryana Board of School Education (HBSE),State


In [255]:
df.loc[
    df["conducting_body"].str.contains("Railway Recruitment Board", case=False, na=False),
    "organization_level"
] = "National"

In [260]:
df[df["conducting_body"].str.contains("Railway Recruitment Board", case=False, na=False)][
    ["exam_name", "conducting_body", "organization_level"]
]

,exam_name,conducting_body,organization_level
4,RRB Mumbai 2010 recruitment exam (Assistant Lo...,"Railway Recruitment Board (RRB), Mumbai",National


In [261]:
df["state"] = df["state"].replace({
    "All India": "Multiple States"
})

df["region"] = df["region"].replace({
    "National": "Multiple Regions"
})

In [262]:
print(df["state"].unique())
print(df["region"].unique())

['Multiple States' 'Himachal Pradesh' 'Madhya Pradesh'
 'Andaman and Nicobar Islands' 'Uttar Pradesh' 'Chhattisgarh' 'Delhi'
 'Karnataka' 'Jammu & Kashmir' 'Assam' 'Rajasthan' 'Manipur' 'Haryana'
 'West Bengal' 'Uttarakhand' 'Bihar/Jharkhand' 'Gujarat' 'Maharashtra'
 'Punjab' 'Tamil Nadu' 'Andhra Pradesh' 'Bihar' 'Arunachal Pradesh'
 'Telangana' 'Odisha' 'Jharkhand']
['Multiple Regions' 'North' 'Central' 'Unknown' 'South' 'North-East'
 'East' 'West']


In [263]:
df.to_csv("../data/cleaned/paper_leaks_cleaned.csv", index=False)

In [265]:
import os
os.path.exists("../data/cleaned/paper_leaks_cleaned.csv")

True

In [266]:
df["conducting_body"] = df["conducting_body"].replace({
    "CBSE (Central Board of Secondary Education)": "CBSE"
})

In [267]:
df["conducting_body"].sort_values().unique()

array(['AIIMS New Delhi',
       'All India Institute of Medical Sciences (AIIMS)',
       'Andhra Pradesh Board of Secondary Education / School Education Dept',
       'Arunachal Pradesh Public Service Commission (APPSC)',
       'Assam Public Service Commission (APSC)',
       'Baba Farid University of Health Sciences (BFUHS)',
       'Bihar Public Service Commission (BPSC)',
       'Bihar Staff Selection Commission (BSSC)',
       'Board of Secondary Education Rajasthan (BSER/RBSE)', 'CBSE',
       'Central Board of Secondary Education (CBSE)',
       'Central Selection Board of Constable (CSBC)',
       'Chhattisgarh Board of Secondary Education (CGBSE)',
       'Chhattisgarh Professional Examination Board (CG Vyapam/CGPEB)',
       'Chhattisgarh Public Service Commission (CGPSC)',
       'Delhi Subordinate Services Selection Board (DSSSB)',
       'Dept. of Pre-University Education, Karnataka',
       'East Central Railway / Railway Board',
       'Gujarat Panchayat Service Select

In [268]:
df["conducting_body"] = df["conducting_body"].replace({
    "CBSE (Central Board of Secondary Education)": "CBSE",
    "National Testing Agency (NTA)": "NTA",
    "Union Public Service Commission (UPSC)": "UPSC",
    "Staff Selection Commission (SSC)": "SSC"
})

In [270]:
df.to_csv("../data/cleaned/paper_leaks_cleaned.csv", index=False)

In [271]:
df["conducting_body"] = df["conducting_body"].replace(
    "CBSE (Central Board of Secondary Education)",
    "CBSE"
)

In [272]:
print(df["conducting_body"].value_counts())

conducting_body
Vyapam (MPPEB)                                                   6
CBSE                                                             5
Central Board of Secondary Education (CBSE)                      4
Rajasthan Staff Selection Board (RSMSSB)                         4
Uttarakhand Subordinate Service Selection Commission (UKSSSC)    4
                                                                ..
Maharashtra Housing and Area Development Authority (MHADA)       1
Tamil Nadu School Education Department                           1
Rajasthan High Court                                             1
Gujarat University / State Forest Department                     1
Chhattisgarh Board of Secondary Education (CGBSE)                1
Name: count, Length: 69, dtype: int64


In [273]:
df["conducting_body"] = df["conducting_body"].replace({
    "Central Board of Secondary Education (CBSE)": "CBSE"
})

In [274]:
df["conducting_body"].value_counts()

conducting_body
CBSE                                                             9
Vyapam (MPPEB)                                                   6
NTA                                                              4
Rajasthan Staff Selection Board (RSMSSB)                         4
Uttarakhand Subordinate Service Selection Commission (UKSSSC)    4
                                                                ..
Tamil Nadu School Education Department                           1
Rajasthan High Court                                             1
Gujarat University / State Forest Department                     1
Himachal Pradesh Police (state government recruitment)           1
Chhattisgarh Board of Secondary Education (CGBSE)                1
Name: count, Length: 68, dtype: int64

In [275]:
df.to_csv("../data/cleaned/paper_leaks_cleaned.csv", index=False)

In [279]:
df.head()

,incident_id,date,era,exam_name,conducting_body,body_type,area,leak_status,action_taken,note,...,affected_group,exam_level,conducting_body_category,exam_mode,decade,state_code,region_code,affected_category,leak_confirmed,repeat_exam
0,PL-0001,2004-04-11,UPA (2004-May2014),All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004,CBSE,Central,All India,Confirmed,Exam cancelled + Arrests-FIR + Probe (CBI/SIT),Physics and chemistry papers leaked in New Del...,...,Unknown,UG,Board,Unknown,2000s,IN,NAT,Unknown,Confirmed,No
1,PL-0002,2006-06-01,UPA (2004-May2014),Himachal Pradesh Combined Pre-Medical Test (HP...,Himachal Pradesh University,State,Himachal Pradesh,Confirmed,Retest + Arrests-FIR,HP-CPMT medical entrance paper leaked via a to...,...,Unknown,UG,University,Unknown,2000s,HP,N,Unknown,Suspected,No
2,PL-0003,2008-01-01,UPA (2004-May2014),Vyapam Patwari Recruitment Examination 2008,Madhya Pradesh Professional Examination Board ...,State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Accused produced forged educational certificat...,...,Unknown,Recruitment,State Recruitment Board,Unknown,2000s,MP,C,Unknown,Confirmed,No
3,PL-0004,2009-07-08,UPA (2004-May2014),MP Pre-Medical Test (PMT) 2009,Vyapam (MPPEB),State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Paid solvers/impersonators sat the exam in pla...,...,<10K,UG,State Recruitment Board,Unknown,2000s,MP,C,Low,Confirmed,No
4,PL-0005,2010-06-06,UPA (2004-May2014),RRB Mumbai 2010 recruitment exam (Assistant Lo...,"Railway Recruitment Board (RRB), Mumbai",Central,All India,Confirmed,Arrests-FIR + Probe (CBI/SIT),CBI registered a case on 15 June 2010 over the...,...,Unknown,Recruitment,Railway,Unknown,2010s,IN,NAT,Unknown,Confirmed,No


In [280]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   incident_id               110 non-null    object        
 1   date                      110 non-null    datetime64[ns]
 2   era                       110 non-null    object        
 3   exam_name                 110 non-null    object        
 4   conducting_body           110 non-null    object        
 5   body_type                 110 non-null    object        
 6   area                      110 non-null    object        
 7   leak_status               110 non-null    object        
 8   action_taken              110 non-null    object        
 9   note                      110 non-null    object        
 10  arrests                   63 non-null     float64       
 11  convictions               10 non-null     float64       
 12  aspirants_affected    

In [281]:
print(df["conducting_body_category"].value_counts())

conducting_body_category
Board                      32
SSC                        19
State PSC                  17
State Recruitment Board    11
University                  7
Police Recruitment          6
Education Department        4
NTA                         4
Government Department       3
Railway                     2
Judiciary                   2
Defence                     1
Medical Board               1
Health Department           1
Name: count, dtype: int64


In [282]:
df["conducting_body_category"] = df["conducting_body_category"].replace(
    "Board",
    "CBSE Board"
)

In [283]:
df.to_csv("../data/cleaned/paper_leaks_cleaned.csv", index=False)

In [284]:
print(df["conducting_body_category"].value_counts())

conducting_body_category
CBSE Board                 32
SSC                        19
State PSC                  17
State Recruitment Board    11
University                  7
Police Recruitment          6
Education Department        4
NTA                         4
Government Department       3
Railway                     2
Judiciary                   2
Defence                     1
Medical Board               1
Health Department           1
Name: count, dtype: int64


In [286]:
print(df.dtypes)

incident_id                         object
date                        datetime64[ns]
era                                 object
exam_name                           object
conducting_body                     object
body_type                           object
area                                object
leak_status                         object
action_taken                        object
note                                object
arrests                            float64
convictions                        float64
aspirants_affected                 float64
linked_deaths                      float64
deaths_note                         object
source_name                         object
source_url                          object
confidence                          object
arrests_status                      object
conviction_status                   object
deaths_status                       object
exam_category                       object
medical_type                        object
engineering

In [287]:
for col in df.columns:
    print(col)

incident_id
date
era
exam_name
conducting_body
body_type
area
leak_status
action_taken
note
arrests
convictions
aspirants_affected
linked_deaths
deaths_note
source_name
source_url
confidence
arrests_status
conviction_status
deaths_status
exam_category
medical_type
engineering_type
forest_type
year
quarter
state
region
organization_level
severity
affected_group
exam_level
conducting_body_category
exam_mode
decade
state_code
region_code
affected_category
leak_confirmed
repeat_exam


In [288]:
df.columns.to_series().to_csv("columns.csv", index=False)

In [289]:
columns.csv

NameError: name 'columns' is not defined

In [292]:
df.columns.to_series().to_csv("columns.csv", index=False)

In [293]:
df.dtypes.to_csv("dtypes.csv")

In [295]:
df.head()


,incident_id,date,era,exam_name,conducting_body,body_type,area,leak_status,action_taken,note,...,affected_group,exam_level,conducting_body_category,exam_mode,decade,state_code,region_code,affected_category,leak_confirmed,repeat_exam
0,PL-0001,2004-04-11,UPA (2004-May2014),All India Pre-Medical Test (AIPMT/CBSE-PMT) 2004,CBSE,Central,All India,Confirmed,Exam cancelled + Arrests-FIR + Probe (CBI/SIT),Physics and chemistry papers leaked in New Del...,...,Unknown,UG,CBSE Board,Unknown,2000s,IN,NAT,Unknown,Confirmed,No
1,PL-0002,2006-06-01,UPA (2004-May2014),Himachal Pradesh Combined Pre-Medical Test (HP...,Himachal Pradesh University,State,Himachal Pradesh,Confirmed,Retest + Arrests-FIR,HP-CPMT medical entrance paper leaked via a to...,...,Unknown,UG,University,Unknown,2000s,HP,N,Unknown,Suspected,No
2,PL-0003,2008-01-01,UPA (2004-May2014),Vyapam Patwari Recruitment Examination 2008,Madhya Pradesh Professional Examination Board ...,State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Accused produced forged educational certificat...,...,Unknown,Recruitment,State Recruitment Board,Unknown,2000s,MP,C,Unknown,Confirmed,No
3,PL-0004,2009-07-08,UPA (2004-May2014),MP Pre-Medical Test (PMT) 2009,Vyapam (MPPEB),State,Madhya Pradesh,Confirmed,Arrests-FIR + Probe (CBI/SIT),Paid solvers/impersonators sat the exam in pla...,...,<10K,UG,State Recruitment Board,Unknown,2000s,MP,C,Low,Confirmed,No
4,PL-0005,2010-06-06,UPA (2004-May2014),RRB Mumbai 2010 recruitment exam (Assistant Lo...,"Railway Recruitment Board (RRB), Mumbai",Central,All India,Confirmed,Arrests-FIR + Probe (CBI/SIT),CBI registered a case on 15 June 2010 over the...,...,Unknown,Recruitment,Railway,Unknown,2010s,IN,NAT,Unknown,Confirmed,No


In [296]:
print(df["incident_id"].is_unique)

True
